# Week 38, Day 3: Project: persistence layer

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w038d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('errors.py', 'w', encoding='utf-8') as f:
    f.write('"""Every error the library raises on purpose."""\n\n\nclass LibraryError(Exception):\n    """Base class for the library\'s own errors."""\n\n\nclass InvalidInput(LibraryError, ValueError):\n    """Something typed or loaded doesn\'t make sense."""\n\n\nclass NotFound(LibraryError, LookupError):\n    """There\'s no book or member with that id."""\n\n\nclass NotAllowed(LibraryError):\n    """The library\'s rules don\'t allow this right now."""\n\n\nclass StorageError(LibraryError):\n    """The library\'s data couldn\'t be read or saved."""\n')
with open('values.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and checking the values people type, and showing money."""\nfrom errors import InvalidInput\n\nKINDS = {"B": "book", "M": "member"}\n\n\ndef clean_isbn(text):\n    """\'978-0-14-118776-1\' as \'9780141187761\'. Raises InvalidInput."""\n    shown = text.strip()\n    digits = shown.replace("-", "").replace(" ", "")\n    if len(digits) != 13 or not (digits.isascii() and digits.isdigit()):\n        raise InvalidInput(f"an ISBN has 13 digits: {shown!r}")\n    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(digits))\n    if total % 10 != 0:\n        raise InvalidInput(f"wrong check digit: {shown!r}")\n    return digits\n\n\ndef clean_text(text, what, longest):\n    """\'  Things   Fall Apart \' as \'Things Fall Apart\'. Raises InvalidInput."""\n    cleaned = " ".join(text.split())\n    if not cleaned:\n        raise InvalidInput(f"{what} is missing")\n    if len(cleaned) > longest:\n        raise InvalidInput(f"{what} is longer than {longest} characters")\n    return cleaned\n\n\ndef parse_id(text, kind):\n    """\' b12 \' as \'B12\', for kind \'B\'. Raises InvalidInput."""\n    cleaned = text.strip().upper()\n    number = cleaned[1:]\n    if not cleaned.startswith(kind) or not (number.isascii() and number.isdigit()) or number.startswith("0"):\n        raise InvalidInput(f"not a {KINDS[kind]} id: {text.strip()!r}")\n    return cleaned\n\n\ndef parse_pence(text):\n    """\'£1.20\' or \'0.6\' as whole pence. Raises InvalidInput."""\n    shown = text.strip()\n    pounds, dot, pence = shown.removeprefix("£").partition(".")\n    if (not (pounds.isascii() and pounds.isdigit())\n            or (dot and not (pence.isascii() and pence.isdigit() and len(pence) <= 2))):\n        raise InvalidInput(f"not an amount: {shown!r}")\n    return int(pounds) * 100 + (int(pence.ljust(2, "0")) if dot else 0)\n\n\ndef format_pence(pence):\n    """60 as \'£0.60\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef check_all(**checks):\n    """Run each check (a function of no arguments); return their results by name, or raise one\n    InvalidInput listing every problem, in order."""\n    results, problems = {}, []\n    for name, check in checks.items():\n        try:\n            results[name] = check()\n        except InvalidInput as err:\n            problems.append(str(err))\n    if problems:\n        raise InvalidInput("; ".join(problems))\n    return results\n')
with open('domain.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s things and rules. No input, output or storage here."""\nfrom dataclasses import dataclass, field\nfrom datetime import date, timedelta\n\nfrom errors import InvalidInput, NotAllowed\nfrom values import format_pence\n\nLOAN_DAYS = 21\nFINE_PER_DAY = 20      # pence for each day late\nMAX_FINE = 500         # the most one loan can cost\nFINE_LIMIT = 500       # owing this much stops a member borrowing\n\n\n@dataclass\nclass Book:\n    book_id: str\n    isbn: str\n    title: str\n    author: str\n    borrower: str | None = None\n    holds: list[str] = field(default_factory=list)\n\n    @property\n    def on_loan(self):\n        return self.borrower is not None\n\n    def describe(self):\n        return f"{self.book_id}: {self.title} by {self.author}"\n\n\n@dataclass\nclass Member:\n    member_id: str\n    name: str\n    max_loans: int = 3\n    owes: int = 0\n\n\n@dataclass\nclass Loan:\n    loan_id: int\n    book_id: str\n    member_id: str\n    out: date\n    due: date\n    returned: date | None = None\n\n    @property\n    def is_open(self):\n        return self.returned is None\n\n    def days_late(self, today):\n        end = self.returned or today\n        return max(0, (end - self.due).days)\n\n    def fine(self, today):\n        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)\n\n\ndef check_can_borrow(member, book, open_loans, today):\n    """Raise NotAllowed with the first reason the member can\'t borrow the book today."""\n    if book.on_loan:\n        raise NotAllowed(f"{book.book_id} is on loan")\n    if book.holds and book.holds[0] != member.member_id:\n        raise NotAllowed(f"{book.book_id} is being kept for someone else")\n    if len(open_loans) >= member.max_loans:\n        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")\n    if any(loan.days_late(today) for loan in open_loans):\n        raise NotAllowed(f"{member.name} has an overdue book")\n    if member.owes >= FINE_LIMIT:\n        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")\n\n\ndef lend(book, member, open_loans, today, loan_id):\n    """Lend the book, or raise NotAllowed. Returns the new Loan."""\n    check_can_borrow(member, book, open_loans, today)\n    if book.holds:\n        book.holds.pop(0)\n    book.borrower = member.member_id\n    return Loan(loan_id, book.book_id, member.member_id, today, today + timedelta(days=LOAN_DAYS))\n\n\ndef give_back(book, loan, member, today):\n    """Close the loan and add any fine to what the member owes. Returns the fine."""\n    assert loan.is_open and loan.book_id == book.book_id and loan.member_id == member.member_id\n    loan.returned = today\n    book.borrower = None\n    fine = loan.fine(today)\n    member.owes += fine\n    return fine\n\n\ndef reserve(book, member):\n    """Join the queue for a book. Returns the member\'s place in it."""\n    if not book.on_loan and not book.holds:\n        raise NotAllowed(f"{book.book_id} is on the shelf: borrow it instead")\n    if book.borrower == member.member_id:\n        raise NotAllowed(f"{member.name} already has {book.book_id}")\n    if member.member_id in book.holds:\n        raise NotAllowed(f"{member.name} is already waiting for {book.book_id}")\n    book.holds.append(member.member_id)\n    return len(book.holds)\n\n\ndef pay(member, pence):\n    """Take a payment towards the member\'s fines. Returns what they still owe."""\n    if pence <= 0:\n        raise InvalidInput("a payment must be more than £0")\n    if pence > member.owes:\n        raise NotAllowed(f"{member.name} only owes {format_pence(member.owes)}")\n    member.owes -= pence\n    return member.owes\n')

### Exercise: Step 13: a store in memory

Write `MemoryStore`. It starts with `books` and `members` (empty dictionaries from id to object), `loans` (an empty list) and `counters` (`{"B": 0, "M": 0, "L": 0}`). Its methods:

- `next_id(kind)` adds 1 to that counter and returns the new id: `"B1"`, `"B2"`, ... for `"B"`, `"M1"`, ... for `"M"`, and plain numbers 1, 2, ... for `"L"` (loans);
- `add_book(book)`, `add_member(member)` and `add_loan(loan)`;
- `book(book_id)` and `member(member_id)`, which return the object or raise `NotFound("no book B9")` / `NotFound("no member M9")` (hiding the `KeyError` with `from None`: the message says it all).

In [ ]:
import json
import logging
import os
from dataclasses import asdict
from datetime import date

from domain import Book, Loan, Member
from errors import NotFound, StorageError

log = logging.getLogger("shelf.store")

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\ndef sample(store):\n    store.add_book(Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\", borrower=\"M1\", holds=[\"M2\"]))\n    store.add_book(Book(\"B2\", \"9780435950828\", \"Arrow of God\", \"Chinua Achebe\"))\n    store.add_member(Member(\"M1\", \"Ada Obi\", owes=60))\n    store.add_member(Member(\"M2\", \"Bo Lee\", max_loans=5))\n    store.add_loan(Loan(1, \"B1\", \"M1\", D(2026, 10, 1), D(2026, 10, 22), D(2026, 10, 25)))\n    store.add_loan(Loan(2, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23)))\n    store.counters = {\"B\": 2, \"M\": 2, \"L\": 2}\n    return store\n\ndef test_ids():\n    \"ids\"\n    s = MemoryStore()\n    assert [s.next_id(\"B\"), s.next_id(\"B\"), s.next_id(\"M\"), s.next_id(\"L\"), s.next_id(\"L\"), s.next_id(\"B\")] == [\"B1\", \"B2\", \"M1\", 1, 2, \"B3\"]\n    t = MemoryStore()\n    assert t.next_id(\"B\") == \"B1\", \"each store has its own counters\"\ndef test_lookups():\n    \"adding and finding\"\n    s = sample(MemoryStore())\n    assert s.book(\"B2\").title == \"Arrow of God\" and s.member(\"M1\").name == \"Ada Obi\" and len(s.loans) == 2\n    for call, message in [(lambda: s.book(\"B9\"), \"no book B9\"), (lambda: s.member(\"M3\"), \"no member M3\")]:\n        try:\n            call()\n            assert False, message\n        except NotFound as err:\n            assert str(err) == message and err.__suppress_context__, f\"got {str(err)!r}; hide the KeyError with from None\"")

### Exercise: Step 14: open loans

Add three methods to `MemoryStore`:

- `open_loans(member_id=None)`: the loans not yet returned, in the order they were added; only that member's if a member id is given;
- `open_loan_for(book_id)`: the open loan of that book, or `NotFound("B2 isn't on loan")`;
- `commit()`, which does nothing: a store in memory has nowhere to save. (Tomorrow's service calls it after every change, so that a store that **does** save can.)

In [ ]:
import json
import logging
import os
from dataclasses import asdict
from datetime import date

from domain import Book, Loan, Member
from errors import NotFound, StorageError

log = logging.getLogger("shelf.store")


class MemoryStore:
    """Keeps everything in dictionaries. commit() does nothing: there's nowhere to save."""

    def __init__(self):
        self.books = {}
        self.members = {}
        self.loans = []
        self.counters = {"B": 0, "M": 0, "L": 0}

    def next_id(self, kind):
        """'B1', 'B2', ... for books, 'M1', ... for members, and 1, 2, ... for loans. Never reused."""
        self.counters[kind] += 1
        number = self.counters[kind]
        return number if kind == "L" else f"{kind}{number}"

    def add_book(self, book):
        self.books[book.book_id] = book

    def add_member(self, member):
        self.members[member.member_id] = member

    def add_loan(self, loan):
        self.loans.append(loan)

    def book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise NotFound(f"no book {book_id}") from None

    def member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise NotFound(f"no member {member_id}") from None

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\ndef sample(store):\n    store.add_book(Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\", borrower=\"M1\", holds=[\"M2\"]))\n    store.add_book(Book(\"B2\", \"9780435950828\", \"Arrow of God\", \"Chinua Achebe\"))\n    store.add_member(Member(\"M1\", \"Ada Obi\", owes=60))\n    store.add_member(Member(\"M2\", \"Bo Lee\", max_loans=5))\n    store.add_loan(Loan(1, \"B1\", \"M1\", D(2026, 10, 1), D(2026, 10, 22), D(2026, 10, 25)))\n    store.add_loan(Loan(2, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23)))\n    store.counters = {\"B\": 2, \"M\": 2, \"L\": 2}\n    return store\n\ndef test_open():\n    \"open loans\"\n    s = sample(MemoryStore())\n    s.add_loan(Loan(3, \"B2\", \"M2\", D(2026, 11, 3), D(2026, 11, 24)))\n    assert [l.loan_id for l in s.open_loans()] == [2, 3] and [l.loan_id for l in s.open_loans(\"M2\")] == [3] and s.open_loans(\"M9\") == []\n    assert s.open_loan_for(\"B1\").loan_id == 2 and s.open_loan_for(\"B2\").loan_id == 3\n    s.loans[2].returned = D(2026, 11, 5)\n    try:\n        s.open_loan_for(\"B2\")\n        assert False\n    except NotFound as err:\n        assert str(err) == \"B2 isn't on loan\"\n    assert s.commit() is None")

### Exercise: Step 15: to plain data

JSON stores dictionaries, lists, text and numbers, but not dataclasses or dates. Write `to_data(store)`, returning a dictionary with:

- `"books"` and `"members"`: a list of each object as a dictionary (`dataclasses.asdict`), in the order they were added;
- `"loans"`: each loan as a dictionary, with `out`, `due` and `returned` as ISO text (`returned` stays `None` if the loan is open);
- `"counters"`: a **copy** of the counters.

In [ ]:
import json
import logging
import os
from dataclasses import asdict
from datetime import date

from domain import Book, Loan, Member
from errors import NotFound, StorageError

log = logging.getLogger("shelf.store")


class MemoryStore:
    """Keeps everything in dictionaries. commit() does nothing: there's nowhere to save."""

    def __init__(self):
        self.books = {}
        self.members = {}
        self.loans = []
        self.counters = {"B": 0, "M": 0, "L": 0}

    def next_id(self, kind):
        """'B1', 'B2', ... for books, 'M1', ... for members, and 1, 2, ... for loans. Never reused."""
        self.counters[kind] += 1
        number = self.counters[kind]
        return number if kind == "L" else f"{kind}{number}"

    def add_book(self, book):
        self.books[book.book_id] = book

    def add_member(self, member):
        self.members[member.member_id] = member

    def add_loan(self, loan):
        self.loans.append(loan)

    def book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise NotFound(f"no book {book_id}") from None

    def member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise NotFound(f"no member {member_id}") from None

    def open_loans(self, member_id=None):
        return [loan for loan in self.loans
                if loan.is_open and (member_id is None or loan.member_id == member_id)]

    def open_loan_for(self, book_id):
        for loan in self.loans:
            if loan.is_open and loan.book_id == book_id:
                return loan
        raise NotFound(f"{book_id} isn't on loan")

    def commit(self):
        pass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\ndef sample(store):\n    store.add_book(Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\", borrower=\"M1\", holds=[\"M2\"]))\n    store.add_book(Book(\"B2\", \"9780435950828\", \"Arrow of God\", \"Chinua Achebe\"))\n    store.add_member(Member(\"M1\", \"Ada Obi\", owes=60))\n    store.add_member(Member(\"M2\", \"Bo Lee\", max_loans=5))\n    store.add_loan(Loan(1, \"B1\", \"M1\", D(2026, 10, 1), D(2026, 10, 22), D(2026, 10, 25)))\n    store.add_loan(Loan(2, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23)))\n    store.counters = {\"B\": 2, \"M\": 2, \"L\": 2}\n    return store\n\nimport json as _json\ndef test_data():\n    \"plain data\"\n    s = sample(MemoryStore())\n    data = to_data(s)\n    assert data[\"books\"][0] == {\"book_id\": \"B1\", \"isbn\": \"9780141187761\", \"title\": \"Things Fall Apart\", \"author\": \"Chinua Achebe\", \"borrower\": \"M1\", \"holds\": [\"M2\"]}\n    assert data[\"members\"] == [{\"member_id\": \"M1\", \"name\": \"Ada Obi\", \"max_loans\": 3, \"owes\": 60}, {\"member_id\": \"M2\", \"name\": \"Bo Lee\", \"max_loans\": 5, \"owes\": 0}]\n    assert data[\"loans\"] == [{\"loan_id\": 1, \"book_id\": \"B1\", \"member_id\": \"M1\", \"out\": \"2026-10-01\", \"due\": \"2026-10-22\", \"returned\": \"2026-10-25\"},\n                             {\"loan_id\": 2, \"book_id\": \"B1\", \"member_id\": \"M1\", \"out\": \"2026-11-02\", \"due\": \"2026-11-23\", \"returned\": None}]\n    assert data[\"counters\"] == {\"B\": 2, \"M\": 2, \"L\": 2}\n    data[\"counters\"][\"B\"] = 99\n    assert s.counters[\"B\"] == 2, \"copy the counters\"\n    _json.dumps(data)")

### Exercise: Step 16: back from plain data

Write `fill(store, data)`, the opposite of `to_data`: it adds every book, member and loan in `data` to an empty store, turning the loans' dates back into `date` objects (`date.fromisoformat`), and sets the store's counters. Don't change the dictionaries in `data` (copy a loan's dictionary before replacing its dates). Loading what `to_data` made must give back equal objects.

In [ ]:
import json
import logging
import os
from dataclasses import asdict
from datetime import date

from domain import Book, Loan, Member
from errors import NotFound, StorageError

log = logging.getLogger("shelf.store")


class MemoryStore:
    """Keeps everything in dictionaries. commit() does nothing: there's nowhere to save."""

    def __init__(self):
        self.books = {}
        self.members = {}
        self.loans = []
        self.counters = {"B": 0, "M": 0, "L": 0}

    def next_id(self, kind):
        """'B1', 'B2', ... for books, 'M1', ... for members, and 1, 2, ... for loans. Never reused."""
        self.counters[kind] += 1
        number = self.counters[kind]
        return number if kind == "L" else f"{kind}{number}"

    def add_book(self, book):
        self.books[book.book_id] = book

    def add_member(self, member):
        self.members[member.member_id] = member

    def add_loan(self, loan):
        self.loans.append(loan)

    def book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise NotFound(f"no book {book_id}") from None

    def member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise NotFound(f"no member {member_id}") from None

    def open_loans(self, member_id=None):
        return [loan for loan in self.loans
                if loan.is_open and (member_id is None or loan.member_id == member_id)]

    def open_loan_for(self, book_id):
        for loan in self.loans:
            if loan.is_open and loan.book_id == book_id:
                return loan
        raise NotFound(f"{book_id} isn't on loan")

    def commit(self):
        pass


def to_data(store):
    """Everything in the store as plain dictionaries and lists, ready for JSON."""
    def loan_data(loan):
        d = asdict(loan)
        d["out"], d["due"] = loan.out.isoformat(), loan.due.isoformat()
        d["returned"] = loan.returned.isoformat() if loan.returned else None
        return d
    return {
        "books": [asdict(b) for b in store.books.values()],
        "members": [asdict(m) for m in store.members.values()],
        "loans": [loan_data(loan) for loan in store.loans],
        "counters": dict(store.counters),
    }

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\ndef sample(store):\n    store.add_book(Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\", borrower=\"M1\", holds=[\"M2\"]))\n    store.add_book(Book(\"B2\", \"9780435950828\", \"Arrow of God\", \"Chinua Achebe\"))\n    store.add_member(Member(\"M1\", \"Ada Obi\", owes=60))\n    store.add_member(Member(\"M2\", \"Bo Lee\", max_loans=5))\n    store.add_loan(Loan(1, \"B1\", \"M1\", D(2026, 10, 1), D(2026, 10, 22), D(2026, 10, 25)))\n    store.add_loan(Loan(2, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23)))\n    store.counters = {\"B\": 2, \"M\": 2, \"L\": 2}\n    return store\n\ndef test_round_trip():\n    \"there and back\"\n    s = sample(MemoryStore())\n    data = to_data(s)\n    t = MemoryStore()\n    fill(t, data)\n    assert t.books == s.books and t.members == s.members and t.loans == s.loans and t.counters == s.counters\n    assert isinstance(t.loans[0].due, D) and t.loans[1].returned is None\n    assert data[\"loans\"][0][\"out\"] == \"2026-10-01\", \"don't change the data you were given\"\n    assert t.next_id(\"B\") == \"B3\"\ndef test_bad():\n    \"bad data fails loudly\"\n    for bad in [{\"books\": [{\"book_id\": \"B1\"}], \"members\": [], \"loans\": [], \"counters\": {}},\n                {\"books\": [], \"members\": [], \"loans\": [], \"counters\": {}, \"x\": 1} | {\"loans\": [{\"loan_id\": 1, \"book_id\": \"B1\", \"member_id\": \"M1\", \"out\": \"2026-13-01\", \"due\": \"2026-11-23\", \"returned\": None}]}]:\n        try:\n            fill(MemoryStore(), bad)\n            assert False, \"damaged data should raise\"\n        except (TypeError, ValueError, KeyError):\n            pass")

### Exercise: Step 17: loading from a file

Write `JsonStore`, a subclass of `MemoryStore` made with a path, which loads the file when it's created (in a method `_load`):

- no file yet: log at `INFO` `"no file at PATH yet: starting empty"`, and start empty;
- the file can't be read, or isn't JSON (an `OSError` or a `ValueError`, which includes `json.JSONDecodeError` and `UnicodeDecodeError`): raise `StorageError("can't read PATH")` **from** the original;
- the JSON doesn't have the right shape (`fill` raises `KeyError`, `TypeError` or `ValueError`): raise `StorageError("PATH is damaged")` **from** the original;
- otherwise log at `INFO` `"loaded B books, M members and L loans from PATH"`.

Log with the module's `log`, passing values as arguments. (Saving comes next.)

In [ ]:
import json
import logging
import os
from dataclasses import asdict
from datetime import date

from domain import Book, Loan, Member
from errors import NotFound, StorageError

log = logging.getLogger("shelf.store")


class MemoryStore:
    """Keeps everything in dictionaries. commit() does nothing: there's nowhere to save."""

    def __init__(self):
        self.books = {}
        self.members = {}
        self.loans = []
        self.counters = {"B": 0, "M": 0, "L": 0}

    def next_id(self, kind):
        """'B1', 'B2', ... for books, 'M1', ... for members, and 1, 2, ... for loans. Never reused."""
        self.counters[kind] += 1
        number = self.counters[kind]
        return number if kind == "L" else f"{kind}{number}"

    def add_book(self, book):
        self.books[book.book_id] = book

    def add_member(self, member):
        self.members[member.member_id] = member

    def add_loan(self, loan):
        self.loans.append(loan)

    def book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise NotFound(f"no book {book_id}") from None

    def member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise NotFound(f"no member {member_id}") from None

    def open_loans(self, member_id=None):
        return [loan for loan in self.loans
                if loan.is_open and (member_id is None or loan.member_id == member_id)]

    def open_loan_for(self, book_id):
        for loan in self.loans:
            if loan.is_open and loan.book_id == book_id:
                return loan
        raise NotFound(f"{book_id} isn't on loan")

    def commit(self):
        pass


def to_data(store):
    """Everything in the store as plain dictionaries and lists, ready for JSON."""
    def loan_data(loan):
        d = asdict(loan)
        d["out"], d["due"] = loan.out.isoformat(), loan.due.isoformat()
        d["returned"] = loan.returned.isoformat() if loan.returned else None
        return d
    return {
        "books": [asdict(b) for b in store.books.values()],
        "members": [asdict(m) for m in store.members.values()],
        "loans": [loan_data(loan) for loan in store.loans],
        "counters": dict(store.counters),
    }


def fill(store, data):
    """Load data made by to_data into an empty store."""
    for d in data["books"]:
        store.add_book(Book(**d))
    for d in data["members"]:
        store.add_member(Member(**d))
    for d in data["loans"]:
        d = dict(d)
        d["out"], d["due"] = date.fromisoformat(d["out"]), date.fromisoformat(d["due"])
        d["returned"] = date.fromisoformat(d["returned"]) if d["returned"] else None
        store.add_loan(Loan(**d))
    store.counters = dict(data["counters"])

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\ndef sample(store):\n    store.add_book(Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\", borrower=\"M1\", holds=[\"M2\"]))\n    store.add_book(Book(\"B2\", \"9780435950828\", \"Arrow of God\", \"Chinua Achebe\"))\n    store.add_member(Member(\"M1\", \"Ada Obi\", owes=60))\n    store.add_member(Member(\"M2\", \"Bo Lee\", max_loans=5))\n    store.add_loan(Loan(1, \"B1\", \"M1\", D(2026, 10, 1), D(2026, 10, 22), D(2026, 10, 25)))\n    store.add_loan(Loan(2, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23)))\n    store.counters = {\"B\": 2, \"M\": 2, \"L\": 2}\n    return store\n\nimport json as _json\nimport logging as _logging\nclass _Keep(_logging.Handler):\n    def __init__(self):\n        super().__init__()\n        self.got = []\n    def emit(self, record):\n        self.got.append((record.levelname, record.getMessage()))\ndef keep():\n    h = _Keep()\n    lg = _logging.getLogger(\"shelf.store\")\n    lg.handlers.clear()\n    lg.addHandler(h)\n    lg.setLevel(_logging.DEBUG)\n    lg.propagate = False\n    return h\ndef test_new():\n    \"no file yet\"\n    h = keep()\n    s = JsonStore(\"lib.json\")\n    assert isinstance(s, MemoryStore) and s.books == {} and s.path == \"lib.json\"\n    assert h.got == [(\"INFO\", \"no file at lib.json yet: starting empty\")], h.got\ndef test_load():\n    \"loading\"\n    write_file(\"lib.json\", _json.dumps(to_data(sample(MemoryStore()))))\n    h = keep()\n    s = JsonStore(\"lib.json\")\n    assert s.book(\"B1\").holds == [\"M2\"] and s.loans[0].due == D(2026, 10, 22) and s.next_id(\"M\") == \"M3\"\n    assert h.got == [(\"INFO\", \"loaded 2 books, 2 members and 2 loans from lib.json\")], h.got\ndef test_damaged():\n    \"damaged files\"\n    for text, message, cause in [(\"{not json\", \"can't read lib.json\", ValueError), ('{\"books\": []}', \"lib.json is damaged\", KeyError),\n                                 ('{\"books\": [{\"title\": \"x\"}], \"members\": [], \"loans\": [], \"counters\": {}}', \"lib.json is damaged\", TypeError)]:\n        write_file(\"lib.json\", text)\n        try:\n            JsonStore(\"lib.json\")\n            assert False, f\"{text!r} should raise StorageError\"\n        except StorageError as err:\n            assert str(err) == message and isinstance(err.__cause__, cause), f\"{text!r}: {err!r} from {err.__cause__!r}\"")

### Exercise: Step 18: saving safely

Add `commit()` to `JsonStore`. Save `to_data(self)` as JSON (`indent=1`, `ensure_ascii=False`) safely, as `spend` did in Week 25: write `PATH.tmp`, then `os.replace` it over the real file, and in a `finally` remove the temporary file if it's still there. An `OSError` becomes `StorageError("can't save to PATH")`, chained from the original. After a successful save, log `DEBUG` `"saved PATH"`.

In [ ]:
import json
import logging
import os
from dataclasses import asdict
from datetime import date

from domain import Book, Loan, Member
from errors import NotFound, StorageError

log = logging.getLogger("shelf.store")


class MemoryStore:
    """Keeps everything in dictionaries. commit() does nothing: there's nowhere to save."""

    def __init__(self):
        self.books = {}
        self.members = {}
        self.loans = []
        self.counters = {"B": 0, "M": 0, "L": 0}

    def next_id(self, kind):
        """'B1', 'B2', ... for books, 'M1', ... for members, and 1, 2, ... for loans. Never reused."""
        self.counters[kind] += 1
        number = self.counters[kind]
        return number if kind == "L" else f"{kind}{number}"

    def add_book(self, book):
        self.books[book.book_id] = book

    def add_member(self, member):
        self.members[member.member_id] = member

    def add_loan(self, loan):
        self.loans.append(loan)

    def book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise NotFound(f"no book {book_id}") from None

    def member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise NotFound(f"no member {member_id}") from None

    def open_loans(self, member_id=None):
        return [loan for loan in self.loans
                if loan.is_open and (member_id is None or loan.member_id == member_id)]

    def open_loan_for(self, book_id):
        for loan in self.loans:
            if loan.is_open and loan.book_id == book_id:
                return loan
        raise NotFound(f"{book_id} isn't on loan")

    def commit(self):
        pass


def to_data(store):
    """Everything in the store as plain dictionaries and lists, ready for JSON."""
    def loan_data(loan):
        d = asdict(loan)
        d["out"], d["due"] = loan.out.isoformat(), loan.due.isoformat()
        d["returned"] = loan.returned.isoformat() if loan.returned else None
        return d
    return {
        "books": [asdict(b) for b in store.books.values()],
        "members": [asdict(m) for m in store.members.values()],
        "loans": [loan_data(loan) for loan in store.loans],
        "counters": dict(store.counters),
    }


def fill(store, data):
    """Load data made by to_data into an empty store."""
    for d in data["books"]:
        store.add_book(Book(**d))
    for d in data["members"]:
        store.add_member(Member(**d))
    for d in data["loans"]:
        d = dict(d)
        d["out"], d["due"] = date.fromisoformat(d["out"]), date.fromisoformat(d["due"])
        d["returned"] = date.fromisoformat(d["returned"]) if d["returned"] else None
        store.add_loan(Loan(**d))
    store.counters = dict(data["counters"])


class JsonStore(MemoryStore):
    """A MemoryStore that loads from a JSON file when it's made, and saves on commit()."""

    def __init__(self, path):
        super().__init__()
        self.path = path
        self._load()

    def _load(self):
        try:
            with open(self.path, encoding="utf-8") as f:
                data = json.load(f)
        except FileNotFoundError:
            log.info("no file at %s yet: starting empty", self.path)
            return
        except (OSError, ValueError) as err:
            raise StorageError(f"can't read {self.path}") from err
        try:
            fill(self, data)
        except (KeyError, TypeError, ValueError) as err:
            raise StorageError(f"{self.path} is damaged") from err
        log.info("loaded %d books, %d members and %d loans from %s",
                 len(self.books), len(self.members), len(self.loans), self.path)

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nD = _dt.date\ndef sample(store):\n    store.add_book(Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\", borrower=\"M1\", holds=[\"M2\"]))\n    store.add_book(Book(\"B2\", \"9780435950828\", \"Arrow of God\", \"Chinua Achebe\"))\n    store.add_member(Member(\"M1\", \"Ada Obi\", owes=60))\n    store.add_member(Member(\"M2\", \"Bo Lee\", max_loans=5))\n    store.add_loan(Loan(1, \"B1\", \"M1\", D(2026, 10, 1), D(2026, 10, 22), D(2026, 10, 25)))\n    store.add_loan(Loan(2, \"B1\", \"M1\", D(2026, 11, 2), D(2026, 11, 23)))\n    store.counters = {\"B\": 2, \"M\": 2, \"L\": 2}\n    return store\n\nimport json as _json\nimport os as _os\ndef test_save():\n    \"saving\"\n    s = JsonStore(\"lib.json\")\n    sample(s)\n    s.book(\"B2\").title = \"Arrow of God \u2013 a novel\"\n    s.commit()\n    text = read_file(\"lib.json\")\n    assert \"\u2013 a novel\" in text, \"use ensure_ascii=False\"\n    assert not _os.path.exists(\"lib.json.tmp\")\n    again = JsonStore(\"lib.json\")\n    assert again.books == s.books and again.loans == s.loans and again.counters == s.counters\ndef test_fails():\n    \"a failed save\"\n    s = JsonStore(\"no-such-folder/lib.json\")\n    try:\n        s.commit()\n        assert False, \"the folder doesn't exist\"\n    except StorageError as err:\n        assert str(err) == \"can't save to no-such-folder/lib.json\" and isinstance(err.__cause__, OSError)\ndef test_keeps_old():\n    \"a bug while saving leaves the old file\"\n    s = JsonStore(\"lib.json\")\n    sample(s)\n    s.commit()\n    before = read_file(\"lib.json\")\n    s.book(\"B2\").holds = {\"M1\"}\n    try:\n        s.commit()\n        assert False, \"a set can't be saved as JSON: that's a bug, a TypeError\"\n    except TypeError:\n        pass\n    assert read_file(\"lib.json\") == before and not _os.path.exists(\"lib.json.tmp\")")

In [ ]:
import json
import logging
import os
from dataclasses import asdict
from datetime import date

from domain import Book, Loan, Member, lend
from errors import NotFound, StorageError

log = logging.getLogger("shelf.store")


class MemoryStore:
    """Keeps everything in dictionaries. commit() does nothing: there's nowhere to save."""

    def __init__(self):
        self.books = {}
        self.members = {}
        self.loans = []
        self.counters = {"B": 0, "M": 0, "L": 0}

    def next_id(self, kind):
        """'B1', 'B2', ... for books, 'M1', ... for members, and 1, 2, ... for loans. Never reused."""
        self.counters[kind] += 1
        number = self.counters[kind]
        return number if kind == "L" else f"{kind}{number}"

    def add_book(self, book):
        self.books[book.book_id] = book

    def add_member(self, member):
        self.members[member.member_id] = member

    def add_loan(self, loan):
        self.loans.append(loan)

    def book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise NotFound(f"no book {book_id}") from None

    def member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise NotFound(f"no member {member_id}") from None

    def open_loans(self, member_id=None):
        return [loan for loan in self.loans
                if loan.is_open and (member_id is None or loan.member_id == member_id)]

    def open_loan_for(self, book_id):
        for loan in self.loans:
            if loan.is_open and loan.book_id == book_id:
                return loan
        raise NotFound(f"{book_id} isn't on loan")

    def commit(self):
        pass


def to_data(store):
    """Everything in the store as plain dictionaries and lists, ready for JSON."""
    def loan_data(loan):
        d = asdict(loan)
        d["out"], d["due"] = loan.out.isoformat(), loan.due.isoformat()
        d["returned"] = loan.returned.isoformat() if loan.returned else None
        return d
    return {
        "books": [asdict(b) for b in store.books.values()],
        "members": [asdict(m) for m in store.members.values()],
        "loans": [loan_data(loan) for loan in store.loans],
        "counters": dict(store.counters),
    }


def fill(store, data):
    """Load data made by to_data into an empty store."""
    for d in data["books"]:
        store.add_book(Book(**d))
    for d in data["members"]:
        store.add_member(Member(**d))
    for d in data["loans"]:
        d = dict(d)
        d["out"], d["due"] = date.fromisoformat(d["out"]), date.fromisoformat(d["due"])
        d["returned"] = date.fromisoformat(d["returned"]) if d["returned"] else None
        store.add_loan(Loan(**d))
    store.counters = dict(data["counters"])


class JsonStore(MemoryStore):
    """A MemoryStore that loads from a JSON file when it's made, and saves on commit()."""

    def __init__(self, path):
        super().__init__()
        self.path = path
        self._load()

    def _load(self):
        try:
            with open(self.path, encoding="utf-8") as f:
                data = json.load(f)
        except FileNotFoundError:
            log.info("no file at %s yet: starting empty", self.path)
            return
        except (OSError, ValueError) as err:
            raise StorageError(f"can't read {self.path}") from err
        try:
            fill(self, data)
        except (KeyError, TypeError, ValueError) as err:
            raise StorageError(f"{self.path} is damaged") from err
        log.info("loaded %d books, %d members and %d loans from %s",
                 len(self.books), len(self.members), len(self.loans), self.path)

    def commit(self):
        """Save everything safely: write a temporary file, then replace the old file in one step."""
        temp = self.path + ".tmp"
        try:
            with open(temp, "w", encoding="utf-8") as f:
                json.dump(to_data(self), f, indent=1, ensure_ascii=False)
            os.replace(temp, self.path)
        except OSError as err:
            raise StorageError(f"can't save to {self.path}") from err
        finally:
            if os.path.exists(temp):
                os.remove(temp)
        log.debug("saved %s", self.path)


logging.basicConfig(level=logging.DEBUG, format="%(levelname)s %(name)s: %(message)s", stream=__import__("sys").stdout)

store = JsonStore("library.json")
book = Book(store.next_id("B"), "9780141187761", "Things Fall Apart", "Chinua Achebe")
ada = Member(store.next_id("M"), "Ada Obi")
store.add_book(book)
store.add_member(ada)
store.add_loan(lend(book, ada, store.open_loans(ada.member_id), date(2026, 11, 2), store.next_id("L")))
store.commit()

later = JsonStore("library.json")            # as if the program had been closed and opened again
print(later.book("B1"))
print(later.open_loan_for("B1"))
print("next book id:", later.next_id("B"))

> **Two kinds of failure.** Look at the two errors in `test_keeps_old` and `test_fails`. A missing folder is the **world** going wrong: the store translates it into a `StorageError`, which the interface will turn into a message. A set in the data is the **program** going wrong (nothing a volunteer types can put a set there), so the store lets the `TypeError` through untranslated, and it will crash with a traceback, to be found and fixed. Either way, the atomic save means the old file is untouched. Translate what the world does; let your own bugs be loud.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The domain from yesterday is in `domain.py` below, with `errors.py` and `values.py`. Today's code is `store.py`, the infrastructure layer.

**What the service needs from a store.** Tomorrow's service will use the store through a handful of methods. Written as a `Protocol` (Week 33), that's the store's contract:

```python
class Store(Protocol):
    def next_id(self, kind: str) -> str | int: ...
    def add_book(self, book: Book) -> None: ...
    def add_member(self, member: Member) -> None: ...
    def add_loan(self, loan: Loan) -> None: ...
    def book(self, book_id: str) -> Book: ...              # NotFound if there isn't one
    def member(self, member_id: str) -> Member: ...        # NotFound if there isn't one
    def open_loans(self, member_id: str | None = None) -> list[Loan]: ...
    def open_loan_for(self, book_id: str) -> Loan: ...     # NotFound if it isn't on loan
    def commit(self) -> None: ...                         # make the changes permanent
```

The service changes the objects the store gives it (setting a book's borrower, say) and then calls `commit()`. A `MemoryStore` has nothing to do on commit; a `JsonStore` saves the file. That's the whole difference between them, which is why `JsonStore` can simply inherit from `MemoryStore` (Week 30) and add loading and saving. Tests will use `MemoryStore`, which is fast and needs no files; the real program will use `JsonStore`.

This is a simpler arrangement than Week 36's one repository per kind of object: one store for the whole library, because loans, books and members are always saved together. Choosing the simpler design that does the job is Week 31's KISS.

## Step 13: a store in memory

Notice that ids come from **counters** in the store, not from "the largest id plus one", for the same reason as `spend`'s `next_id`: an id, once used, is never handed out again. And for the first time, the counters live with the data they number, so they're saved with it.

## Step 14: open loans

Loans are never deleted: a returned loan stays as history, with its `returned` date. So the store needs a way to find the loans that are still open.

## Step 15: to plain data

## Step 16: back from plain data

## Step 17: loading from a file

This is the boundary between the program and the outside world, so it's where Week 37's **translation** happens. Above this layer, nobody should need to know that the data is JSON in a file: they catch `StorageError`. The original error is kept as the cause, so the detail is there for whoever has to fix the file.

A missing file isn't an error at all: it's a new library. That's handled here too, and logged, because "I started with an empty library" is exactly the kind of thing someone will later want to know.

## Step 18: saving safely

## Checkpoint

Here's the finished `store.py`, used with the domain the way tomorrow's service will: make a book and a member, lend the book, commit, and then load the library again as a fresh store, as if the program had been restarted. Logging is set to `DEBUG` so you can see the store at work.

## Quiz

## Before you go

Put today's code in `store.py` and commit. Try `JsonStore` on a file you've damaged by hand, and read the traceback: the `StorageError` on top, the original below. Tomorrow: the service with the use cases, the text interface, and the composition root that wires it all together.